## Setup and Imports

In [5]:
import ast
import numpy as np
import pandas as pd
import scipy.sparse as sp
from datetime import datetime, timedelta
import itertools

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.ml.recommendation import ALS
from pyspark.mllib.evaluation import RankingMetrics

from lightfm.data import Dataset
from lightfm import LightFM
from lightfm.evaluation import precision_at_k, recall_at_k

import mlflow
import mlflow.spark

# Initialize MLflow Experiment
mlflow.set_experiment("ALS_LFM")

# 1. Point to the remote server
mlflow.set_tracking_uri("http://gcp-prd-ds-mlflow.prd.wynk.internal/") 

# Initialize Spark
spark = SparkSession.builder \
    .appName("hyperparam_tuning") \
    .config("spark.sql.shuffle.partitions", "4000") \
    .config("spark.executor.memoryOverhead", "4g") \
    .config("spark.sql.files.ignoreCorruptFiles", "true") \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .config("spark.sql.parquet.mergeSchema", "true") \
    .getOrCreate()

26/03/24 10:21:56 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


26/03/24 11:01:35 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 59 for reason Executor for container container_1764236692086_5626_01_000079 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/24 11:01:35 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 18 for reason Executor for container container_1764236692086_5626_01_000022 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.


## Data Pipeline

In [2]:
class RecommendationDataPipeline:
    def __init__(self, spark, config):
        self.spark = spark
        self.config = config
        self.train_df = None
        self.test_df = None
        self.metadata_df = None
        self.indexed_train = None
        self.indexed_test = None
        
    def _get_valid_paths(self, base_path, start_days_ago, end_days_ago):
        base_date = datetime.strptime(self.config['date'], "%Y-%m-%d")
        paths = []
        for i in range(start_days_ago, end_days_ago):
            target_date = (base_date - timedelta(days=i)).strftime("%Y-%m-%d")
            paths.append(f"{base_path}/day={target_date}")
        return paths

    def _read_and_union_paths(self, paths):
        df = None
        for path in paths:
            try:
                temp_df = self.spark.read.parquet(path)
                df = temp_df if df is None else df.unionByName(temp_df)
            except Exception:
                pass # Skipping missing paths
        return df

    def load_raw_data(self):
        print("Loading raw interactions and metadata...")
        test_paths = self._get_valid_paths(self.config['watch_history_path'], 0, self.config['test_days'])
        train_paths = self._get_valid_paths(self.config['watch_history_path'], self.config['test_days'], self.config['test_days'] + self.config['train_days'])
        
        self.test_df = self._read_and_union_paths(test_paths)
        self.train_df = self._read_and_union_paths(train_paths)
        
        # Load Metadata
        tv_df = self.spark.read.parquet(f"{self.config['db_path']}{self.config['date']}/enriched_tv.parquet")
        movie_df = self.spark.read.parquet(f"{self.config['db_path']}{self.config['date']}/enriched_movie.parquet")
        
        def clean_meta(df):
            return (df.filter((F.col('XstreamContentIds') != F.array()) & (F.col("published") == True))
                    .withColumn("item_id_exploded", F.explode("XstreamContentIds")) # Step 1: Explode
                    .select(
                        F.col("item_id_exploded").cast("string").alias("item_id"), # Step 2: Cast
                        "title", 
                        F.col('OriginalLanguage').alias('original_language').cast("string"),
                        "Genres"
                    ))
        self.metadata_df = clean_meta(tv_df).unionByName(clean_meta(movie_df)).distinct()
        

    def process_and_index_data(self):
        print("Filtering users, aggregating playtime, and building indexes...")
        
        # 1. Overlapping Users & Aggregation
        common_users = self.train_df.select("userId").distinct().join(self.test_df.select("userId").distinct(), "userId")
        
        train_filtered = self.train_df.join(common_users, "userId")
        test_filtered = self.test_df.join(common_users, "userId")
        
        train_stats = train_filtered.groupBy("userId", "item_id") \
            .agg(F.sum("total_play_time_sec").alias("total_playtime_combined")) \
            .withColumn("distinct_content_count", F.count("item_id").over(Window.partitionBy("userId"))) \
            .filter(F.col("total_playtime_combined").isNotNull() & ~F.isnan("total_playtime_combined"))
            
        als_input_base = train_stats.filter("distinct_content_count >= 2")
        
        # Re-filter test users based on the >= 2 rule
        valid_users = als_input_base.select("userId").distinct()
        test_filtered = test_filtered.join(valid_users, "userId")
        
        # 2. Build Lookup Tables
        distinct_users = valid_users.rdd.zipWithIndex().toDF(["user_struct", "userIndex"]).select(F.col("user_struct.*"), F.col("userIndex").cast("int"))
        distinct_items = als_input_base.select("item_id").distinct().rdd.zipWithIndex().toDF(["item_struct", "itemIndex"]).select(F.col("item_struct.*"), F.col("itemIndex").cast("int"))
        
        # Break Lineage!
        distinct_users.write.mode("overwrite").parquet(f"{self.config['temp_path']}/user_lookup")
        distinct_items.write.mode("overwrite").parquet(f"{self.config['temp_path']}/item_lookup")
        
        user_lookup = self.spark.read.parquet(f"{self.config['temp_path']}/user_lookup")
        self.item_lookup = self.spark.read.parquet(f"{self.config['temp_path']}/item_lookup")

        # 3. Apply Indexes
        self.indexed_train = als_input_base.join(user_lookup, "userId").join(self.item_lookup, "item_id") \
            .withColumn("playtime_logged", F.log1p("total_playtime_combined")) \
            .select("userIndex", "itemIndex", "playtime_logged").repartition(1000).cache()
            
        self.indexed_test = test_filtered.join(user_lookup, "userId").join(self.item_lookup, "item_id") \
            .withColumn("playtime_logged", F.log1p("total_play_time_sec")) \
            .select("userIndex", "itemIndex", "playtime_logged").cache()

    def get_als_data(self):
        """Returns DataFrames optimized for Spark ALS."""
        ground_truth = self.indexed_test.groupBy("userIndex").agg(F.collect_set("itemIndex").alias("actual_items"))
        return self.indexed_train, ground_truth

    def get_lightfm_data(self, sample_fraction=0.05):
        """Samples, extracts features, and builds Scipy matrices for LightFM."""
        print(f"Preparing LightFM Matrices (Sampling {sample_fraction * 100}% of users)...")
        
        # Scrub test data of seen items
        clean_test = self.indexed_test.join(self.indexed_train.select("userIndex", "itemIndex"), on=["userIndex", "itemIndex"], how="left_anti")
        
        # Sample users to fit in driver memory
        sampled_users = self.indexed_train.select("userIndex").distinct().sample(withReplacement=False, fraction=sample_fraction, seed=42)
        
        train_pdf = self.indexed_train.join(sampled_users, "userIndex").toPandas()
        test_pdf = clean_test.join(sampled_users, "userIndex").toPandas()
        
        items_pdf = self.metadata_df.join(self.item_lookup, "item_id").select("itemIndex", "original_language", "Genres").dropDuplicates(["itemIndex"]).toPandas()
        
        # Clean & extract metadata features
        items_pdf['Genres'] = items_pdf['Genres'].fillna('Unknown').astype(str)
        items_pdf['original_language'] = items_pdf['original_language'].fillna('Unknown').astype(str)
        
        valid_items = np.unique(np.concatenate((train_pdf['itemIndex'], test_pdf['itemIndex'])))
        items_pdf = items_pdf[items_pdf['itemIndex'].isin(valid_items)]
        
        def extract_features(lang, genre_data):
            feats = [str(lang)]
            if genre_data.startswith('['):
                try: genre_data = ast.literal_eval(genre_data)
                except: genre_data = genre_data.replace('[', '').replace(']', '').replace("'", "").replace('"', "").split(',')
            else:
                genre_data = genre_data.split(',')
            feats.extend([g.strip() for g in genre_data])
            return list(set(feats))
            
        items_pdf['clean_features'] = items_pdf.apply(lambda row: extract_features(row['original_language'], row['Genres']), axis=1)
        all_features = list(set([feat for sublist in items_pdf['clean_features'] for feat in sublist]))
        
        # Build LightFM Dataset
        dataset = Dataset()
        all_users = np.unique(np.concatenate((train_pdf['userIndex'], test_pdf['userIndex'])))
        dataset.fit(users=all_users, items=valid_items, item_features=all_features)
        
        train_interact, train_weights = dataset.build_interactions(zip(train_pdf['userIndex'], train_pdf['itemIndex'], train_pdf['playtime_logged']))
        test_interact, _ = dataset.build_interactions(zip(test_pdf['userIndex'], test_pdf['itemIndex'], test_pdf['playtime_logged']))
        item_features = dataset.build_item_features((idx, feats) for idx, feats in zip(items_pdf['itemIndex'], items_pdf['clean_features']))
        
        return train_interact, train_weights, test_interact, item_features

## Run Pipeline

In [3]:
config = {
    "date": "2026-03-22",
    "train_days": 90,
    "test_days": 30,
    "watch_history_path": "gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new",
    "db_path": "gs://wynk-ml-workspace/projects/xstream_nlu/catalog-db/",
    "temp_path": "gs://wynk-ml-workspace/_temp/harshith/als"
}

pipeline = RecommendationDataPipeline(spark, config)
pipeline.load_raw_data()
pipeline.process_and_index_data()

Loading raw interactions and metadata...


26/03/24 10:10:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Filtering users, aggregating playtime, and building indexes...


In [ ]:
def print_memory_diagnostics(pipeline):
    print("Forcing Spark to materialize the caches... (this may take a few minutes)")
    
    # 1. Get exact row counts
    train_rows = pipeline.indexed_train.count()
    test_rows = pipeline.indexed_test.count()
    
    # 2. Calculate Pandas memory (assuming 24 bytes per row overhead)
    bytes_per_row = 24
    train_gb = (train_rows * bytes_per_row) / (1024**3)
    test_gb = (test_rows * bytes_per_row) / (1024**3)
    
    print("-" * 30)
    print(f"Train Rows: {train_rows:,}")
    print(f"Test Rows:  {test_rows:,}")
    print("-" * 30)
    print(f"Expected Pandas Train RAM: {train_gb:.2f} GB")
    print(f"Expected Pandas Test RAM:  {test_gb:.2f} GB")
    print(f"Total Expected Pandas RAM: {(train_gb + test_gb):.2f} GB")
    print("-" * 30)

print_memory_diagnostics(pipeline)

Forcing Spark to materialize the caches... (this may take a few minutes)


26/03/24 10:11:58 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/24 10:11:59 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/24 10:12:04 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/24 10:12:05 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/24 10:

------------------------------
Train Rows: 40,202,404
Test Rows:  31,379,347
------------------------------
Expected Pandas Train RAM: 0.90 GB
Expected Pandas Test RAM:  0.70 GB
Total Expected Pandas RAM: 1.60 GB
------------------------------


26/03/24 10:13:31 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 63 for reason Executor for container container_1764236692086_5626_01_000084 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.


## Hyper Parameter Tuning

In [11]:
def tune_als(train_data, ground_truth, param_grid):
    print("Starting ALS Hyperparameter Tuning...")
    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    for params in combinations:
        with mlflow.start_run(run_name=f"ALS_rank_{params['rank']}_reg_{params['regParam']}"):
            mlflow.log_params(params)
            mlflow.log_param("model_type", "ALS")
            
            als = ALS(
                userCol="userIndex", itemCol="itemIndex", ratingCol="playtime_logged",
                implicitPrefs=True, maxIter=10, coldStartStrategy="drop",
                rank=params['rank'], regParam=params['regParam']
            )
            
            model = als.fit(train_data)
            
            # Evaluate
            user_recs = model.recommendForAllUsers(15)
            user_recs_flat = user_recs.select("userIndex", F.col("recommendations.itemIndex").alias("predicted_items"))
            eval_data = user_recs_flat.join(ground_truth, "userIndex").select("predicted_items", "actual_items").rdd.map(tuple)
            
            metrics = RankingMetrics(eval_data)
            map_metric = metrics.meanAveragePrecision
            precision_15 = metrics.precisionAt(15)
            
            mlflow.log_metrics({
                "MAP": map_metric,
                "Precision_at_15": precision_15,
                "Recall_at_15": metrics.recallAt(15)
            })
            print(f"ALS Params: {params} -> MAP: {map_metric:.4f}, P@15: {precision_15:.4f}")

def tune_lightfm(train_interact, train_weights, test_interact, item_features, param_grid):
    print("Starting LightFM Hyperparameter Tuning...")
    
    # Scrub test interactions locally
    test_csr = test_interact.tocsr()
    train_csr = train_interact.tocsr()
    overlap = test_csr.multiply(train_csr.astype(bool))
    clean_test_interactions = (test_csr - overlap).tocoo()
    
    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    for params in combinations:
        with mlflow.start_run(run_name=f"LFM_comp_{params['no_components']}_lr_{params['learning_rate']}"):
            mlflow.log_params(params)
            mlflow.log_param("model_type", "LightFM")
            
            model = LightFM(
                loss=params['loss'], 
                no_components=params['no_components'],
                learning_rate=params['learning_rate'],
                item_alpha=1e-6, user_alpha=1e-6
            )
            
            model.fit(train_interact, item_features=item_features, sample_weight=train_weights, epochs=10, num_threads=4)
            
            precision = precision_at_k(model, clean_test_interactions, train_interactions=train_interact, item_features=item_features, k=15, num_threads=4).mean()
            recall = recall_at_k(model, clean_test_interactions, train_interactions=train_interact, item_features=item_features, k=15, num_threads=4).mean()
            
            mlflow.log_metrics({
                "Precision_at_15": float(precision),
                "Recall_at_15": float(recall)
            })
            print(f"LightFM Params: {params} -> P@15: {precision:.4f}, R@15: {recall:.4f}")

## Execute Tuning

In [12]:
# 1. Fetch data formats
als_train, als_ground_truth = pipeline.get_als_data()
lfm_train, lfm_weights, lfm_test, lfm_item_feats = pipeline.get_lightfm_data(sample_fraction=0.05)

# 2. Define Grids
als_param_grid = {
    "rank": [10, 20, 50, 100, 200],
    "regParam": [0.01, 0.1]
}

lfm_param_grid = {
    "no_components": [20, 64, 128],
    "learning_rate": [0.01, 0.05],
    "loss": ["warp"]
}

# 3. Run Grids
tune_als(als_train, als_ground_truth, als_param_grid)
tune_lightfm(lfm_train, lfm_weights, lfm_test, lfm_item_feats, lfm_param_grid)

print("Tuning complete! View results on your MLflow UI.")

Preparing LightFM Matrices (Sampling 5.0% of users)...


26/03/23 09:06:04 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 09:06:05 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 09:06:35 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 09:06:36 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 09:

Starting ALS Hyperparameter Tuning...


26/03/23 09:11:07 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 20 for reason Executor for container container_1764236692086_5553_01_000033 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/23 09:11:07 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 17 for reason Executor for container container_1764236692086_5553_01_000027 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/23 09:11:07 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 19 for reason Executor for container container_1764236692086_5553_01_000029 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/23 09:11:07 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 18 for reason Executor for container container_1764236692086_5

ALS Params: {'rank': 10, 'regParam': 0.01} -> MAP: 0.1174, P@15: 0.0731


...

LightFM Params: {'no_components': 20, 'learning_rate': 0.01, 'loss': 'warp'} -> P@15: 0.0100, R@15: 0.0467


2026/03/23 11:07:01 INFO mlflow.tracking._tracking_service.client: 🏃 View run LFM_comp_20_lr_0.05 at: http://gcp-prd-ds-mlflow.prd.wynk.internal/#/experiments/20/runs/72946bdec5a048c3af192b0c2dfdac62.
2026/03/23 11:07:01 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://gcp-prd-ds-mlflow.prd.wynk.internal/#/experiments/20.


LightFM Params: {'no_components': 20, 'learning_rate': 0.05, 'loss': 'warp'} -> P@15: 0.0136, R@15: 0.0613


2026/03/23 11:07:38 INFO mlflow.tracking._tracking_service.client: 🏃 View run LFM_comp_64_lr_0.01 at: http://gcp-prd-ds-mlflow.prd.wynk.internal/#/experiments/20/runs/b402a71b52f042a9a2394f04de2b39f8.
2026/03/23 11:07:38 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://gcp-prd-ds-mlflow.prd.wynk.internal/#/experiments/20.


KeyboardInterrupt: 

In [ ]:
import ast
import numpy as np
import pandas as pd
import scipy.sparse as sp
from datetime import datetime, timedelta
import itertools

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.ml.recommendation import ALS
from pyspark.mllib.evaluation import RankingMetrics

from lightfm.data import Dataset
from lightfm import LightFM
from lightfm.evaluation import precision_at_k, recall_at_k

import mlflow
import mlflow.spark


# 1. Point to the remote server
mlflow.set_tracking_uri("http://gcp-prd-ds-mlflow.prd.wynk.internal/") 

# 3. Optional: Verify it's connected before starting the loop
# Initialize MLflow Experiment
mlflow.set_experiment("ALS_LFM")
experiment_name = "ALS_LFM"
exp = mlflow.get_experiment_by_name(experiment_name)
print(f"Connected to MLflow Experiment: {exp.name} with ID: {exp.experiment_id}")

# Initialize Spark
spark = SparkSession.builder \
    .appName("lightfm_tuning2") \
    .config("spark.sql.shuffle.partitions", "4000") \
    .config("spark.executor.memoryOverhead", "4g") \
    .config("spark.sql.files.ignoreCorruptFiles", "true") \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .config("spark.sql.parquet.mergeSchema", "true") \
    .getOrCreate()

class RecommendationDataPipeline:
    def __init__(self, spark, config):
        self.spark = spark
        self.config = config
        self.train_df = None
        self.test_df = None
        self.metadata_df = None
        self.indexed_train = None
        self.indexed_test = None
        
    def _get_valid_paths(self, base_path, start_days_ago, end_days_ago):
        base_date = datetime.strptime(self.config['date'], "%Y-%m-%d")
        paths = []
        for i in range(start_days_ago, end_days_ago):
            target_date = (base_date - timedelta(days=i)).strftime("%Y-%m-%d")
            paths.append(f"{base_path}/day={target_date}")
        return paths

    def _read_and_union_paths(self, paths):
        df = None
        for path in paths:
            try:
                temp_df = self.spark.read.parquet(path)
                df = temp_df if df is None else df.unionByName(temp_df)
            except Exception:
                pass # Skipping missing paths
        return df

    def load_raw_data(self):
        print("Loading raw interactions and metadata...")
        test_paths = self._get_valid_paths(self.config['watch_history_path'], 0, self.config['test_days'])
        train_paths = self._get_valid_paths(self.config['watch_history_path'], self.config['test_days'], self.config['test_days'] + self.config['train_days'])
        
        self.test_df = self._read_and_union_paths(test_paths)
        self.train_df = self._read_and_union_paths(train_paths)
        
        # Load Metadata
        tv_df = self.spark.read.parquet(f"{self.config['db_path']}{self.config['date']}/enriched_tv.parquet")
        movie_df = self.spark.read.parquet(f"{self.config['db_path']}{self.config['date']}/enriched_movie.parquet")
        
        def clean_meta(df):
            return (df.filter((F.col('XstreamContentIds') != F.array()) & (F.col("published") == True))
                    .withColumn("item_id_exploded", F.explode("XstreamContentIds")) # Step 1: Explode
                    .select(
                        F.col("item_id_exploded").cast("string").alias("item_id"), # Step 2: Cast
                        "title", 
                        F.col('OriginalLanguage').alias('original_language').cast("string"),
                        "Genres"
                    ))
        self.metadata_df = clean_meta(tv_df).unionByName(clean_meta(movie_df)).distinct()
        

    def process_and_index_data(self):
        print("Filtering users, aggregating playtime, and building indexes...")
        
        # 1. Overlapping Users & Aggregation
        common_users = self.train_df.select("userId").distinct().join(self.test_df.select("userId").distinct(), "userId")
        
        train_filtered = self.train_df.join(common_users, "userId")
        test_filtered = self.test_df.join(common_users, "userId")
        
        train_stats = train_filtered.groupBy("userId", "item_id") \
            .agg(F.sum("total_play_time_sec").alias("total_playtime_combined")) \
            .withColumn("distinct_content_count", F.count("item_id").over(Window.partitionBy("userId"))) \
            .filter(F.col("total_playtime_combined").isNotNull() & ~F.isnan("total_playtime_combined"))
            
        als_input_base = train_stats.filter("distinct_content_count >= 2")
        
        # Re-filter test users based on the >= 2 rule
        valid_users = als_input_base.select("userId").distinct()
        test_filtered = test_filtered.join(valid_users, "userId")
        
        # 2. Build Lookup Tables
        distinct_users = valid_users.rdd.zipWithIndex().toDF(["user_struct", "userIndex"]).select(F.col("user_struct.*"), F.col("userIndex").cast("int"))
        distinct_items = als_input_base.select("item_id").distinct().rdd.zipWithIndex().toDF(["item_struct", "itemIndex"]).select(F.col("item_struct.*"), F.col("itemIndex").cast("int"))
        
        # Break Lineage!
        distinct_users.write.mode("overwrite").parquet(f"{self.config['temp_path']}/user_lookup")
        distinct_items.write.mode("overwrite").parquet(f"{self.config['temp_path']}/item_lookup")
        
        user_lookup = self.spark.read.parquet(f"{self.config['temp_path']}/user_lookup")
        self.item_lookup = self.spark.read.parquet(f"{self.config['temp_path']}/item_lookup")

        # 3. Apply Indexes
        self.indexed_train = als_input_base.join(user_lookup, "userId").join(self.item_lookup, "item_id") \
            .withColumn("playtime_logged", F.log1p("total_playtime_combined")) \
            .select("userIndex", "itemIndex", "playtime_logged").repartition(1000).cache()
            
        self.indexed_test = test_filtered.join(user_lookup, "userId").join(self.item_lookup, "item_id") \
            .withColumn("playtime_logged", F.log1p("total_play_time_sec")) \
            .select("userIndex", "itemIndex", "playtime_logged").cache()

    def get_als_data(self):
        """Returns DataFrames optimized for Spark ALS."""
        ground_truth = self.indexed_test.groupBy("userIndex").agg(F.collect_set("itemIndex").alias("actual_items"))
        return self.indexed_train, ground_truth

    def get_lightfm_data(self, sample_fraction=0.05):
        """Samples, extracts features, and builds Scipy matrices for LightFM."""
        print(f"Preparing LightFM Matrices (Sampling {sample_fraction * 100}% of users)...")
        
        # Scrub test data of seen items
        clean_test = self.indexed_test.join(self.indexed_train.select("userIndex", "itemIndex"), on=["userIndex", "itemIndex"], how="left_anti")
        
        # Sample users to fit in driver memory
        sampled_users = self.indexed_train.select("userIndex").distinct().sample(withReplacement=False, fraction=sample_fraction, seed=42)
        
        train_pdf = self.indexed_train.join(sampled_users, "userIndex").toPandas()
        test_pdf = clean_test.join(sampled_users, "userIndex").toPandas()
        
        items_pdf = self.metadata_df.join(self.item_lookup, "item_id").select("itemIndex", "original_language", "Genres").dropDuplicates(["itemIndex"]).toPandas()
        
        # Clean & extract metadata features
        items_pdf['Genres'] = items_pdf['Genres'].fillna('Unknown').astype(str)
        items_pdf['original_language'] = items_pdf['original_language'].fillna('Unknown').astype(str)
        
        valid_items = np.unique(np.concatenate((train_pdf['itemIndex'], test_pdf['itemIndex'])))
        items_pdf = items_pdf[items_pdf['itemIndex'].isin(valid_items)]
        
        def extract_features(lang, genre_data):
            feats = [str(lang)]
            if genre_data.startswith('['):
                try: genre_data = ast.literal_eval(genre_data)
                except: genre_data = genre_data.replace('[', '').replace(']', '').replace("'", "").replace('"', "").split(',')
            else:
                genre_data = genre_data.split(',')
            feats.extend([g.strip() for g in genre_data])
            return list(set(feats))
            
        items_pdf['clean_features'] = items_pdf.apply(lambda row: extract_features(row['original_language'], row['Genres']), axis=1)
        all_features = list(set([feat for sublist in items_pdf['clean_features'] for feat in sublist]))
        
        # Build LightFM Dataset
        dataset = Dataset()
        all_users = np.unique(np.concatenate((train_pdf['userIndex'], test_pdf['userIndex'])))
        dataset.fit(users=all_users, items=valid_items, item_features=all_features)
        
        train_interact, train_weights = dataset.build_interactions(zip(train_pdf['userIndex'], train_pdf['itemIndex'], train_pdf['playtime_logged']))
        test_interact, _ = dataset.build_interactions(zip(test_pdf['userIndex'], test_pdf['itemIndex'], test_pdf['playtime_logged']))
        item_features = dataset.build_item_features((idx, feats) for idx, feats in zip(items_pdf['itemIndex'], items_pdf['clean_features']))
        
        return train_interact, train_weights, test_interact, item_features
    
config = {
    "date": "2026-03-22",
    "train_days": 90,
    "test_days": 30,
    "watch_history_path": "gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new",
    "db_path": "gs://wynk-ml-workspace/projects/xstream_nlu/catalog-db/",
    "temp_path": "gs://wynk-ml-workspace/_temp/harshith/als"
}

pipeline = RecommendationDataPipeline(spark, config)
pipeline.load_raw_data()
pipeline.process_and_index_data()

def tune_lightfm(train_interact, train_weights, test_interact, item_features, param_grid):
    print("Starting LightFM Hyperparameter Tuning...")
    
    # Scrub test interactions locally
    test_csr = test_interact.tocsr()
    train_csr = train_interact.tocsr()
    overlap = test_csr.multiply(train_csr.astype(bool))
    clean_test_interactions = (test_csr - overlap).tocoo()
    
    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    for params in combinations:
        with mlflow.start_run(run_name=f"LFM_comp_{params['no_components']}_lr_{params['learning_rate']}"):
            mlflow.log_params(params)
            mlflow.log_param("model_type", "LightFM")
            
            model = LightFM(
                loss=params['loss'], 
                no_components=params['no_components'],
                learning_rate=params['learning_rate'],
                item_alpha=params['alpha'], user_alpha=params['alpha']
            )
            
            model.fit(train_interact, item_features=item_features, sample_weight=train_weights, epochs=params['epochs'], num_threads=4)
            
            precision = precision_at_k(model, clean_test_interactions, train_interactions=train_interact, item_features=item_features, k=15, num_threads=4).mean()
            recall = recall_at_k(model, clean_test_interactions, train_interactions=train_interact, item_features=item_features, k=15, num_threads=4).mean()
            
            mlflow.log_metrics({
                "Precision_at_15": float(precision),
                "Recall_at_15": float(recall)
            })
            print(f"LightFM Params: {params} -> P@15: {precision:.4f}, R@15: {recall:.4f}")

lfm_train, lfm_weights, lfm_test, lfm_item_feats = pipeline.get_lightfm_data(sample_fraction=0.05)


lfm_param_grid = {
    "no_components": [20, 64, 128],
    "learning_rate": [0.01, 0.05],
    "loss": ["warp"],
    "epochs": [10,25,40],  # Fixed for tuning, can be expanded if needed
    "alpha": [1e-6, 1e-5, 1e-4],  # Regularization for items
}
tune_lightfm(lfm_train, lfm_weights, lfm_test, lfm_item_feats, lfm_param_grid)

print("Tuning complete! View results on your MLflow UI.")

Connected to MLflow Experiment: ALS_LFM with ID: 20
Loading raw interactions and metadata...
Filtering users, aggregating playtime, and building indexes...


26/03/23 16:39:15 WARN CacheManager: Asked to cache already cached data.         623]]
26/03/23 16:39:15 WARN CacheManager: Asked to cache already cached data.


Preparing LightFM Matrices (Sampling 5.0% of users)...


26/03/23 16:39:16 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 16:39:16 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 16:39:41 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 16:39:42 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 16:

Starting LightFM Hyperparameter Tuning...


...

LightFM Params: {'no_components': 20, 'learning_rate': 0.01, 'loss': 'warp', 'epochs': 40, 'alpha': 0.0001} -> P@15: 0.0108, R@15: 0.0502


2026/03/23 17:10:29 INFO mlflow.tracking._tracking_service.client: 🏃 View run LFM_comp_20_lr_0.05 at: http://gcp-prd-ds-mlflow.prd.wynk.internal/#/experiments/20/runs/70fbe138e2a8451499f457cda67df851.
2026/03/23 17:10:29 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://gcp-prd-ds-mlflow.prd.wynk.internal/#/experiments/20.


LightFM Params: {'no_components': 20, 'learning_rate': 0.05, 'loss': 'warp', 'epochs': 10, 'alpha': 1e-06} -> P@15: 0.0145, R@15: 0.0646


2026/03/23 17:12:19 INFO mlflow.tracking._tracking_service.client: 🏃 View run LFM_comp_20_lr_0.05 at: http://gcp-prd-ds-mlflow.prd.wynk.internal/#/experiments/20/runs/90c445d8866d45b4b7371d7196f4ec71.
2026/03/23 17:12:19 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://gcp-prd-ds-mlflow.prd.wynk.internal/#/experiments/20.


LightFM Params: {'no_components': 20, 'learning_rate': 0.05, 'loss': 'warp', 'epochs': 10, 'alpha': 1e-05} -> P@15: 0.0136, R@15: 0.0606


In [ ]:
/opt/conda/miniconda3/bin/spark-submit --master yarn --conf spark.pyspark.python=/opt/conda/default/bin/python

In [ ]:
/opt/conda/miniconda3/bin/spark-submit --master yarn --conf spark.pyspark.python=/opt/conda/default/bin/pythonwork/RecSys/LFM_hypTun.py

In [ ]:
/opt/conda/miniconda3/bin/spark-submit --master yarn --conf spark.pyspark.python=/opt/conda/default/bin/python --conf spark.maximizeResourceAllocation=true --conf spark.dynamicAllocation.enabled=true /home/B0336827/work/RecSys/LFM_hypTun.py

In [6]:
import sys
print(sys.executable)

/opt/conda/default/bin/python


26/03/24 11:21:09 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 60 for reason Executor for container container_1764236692086_5626_01_000081 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/24 11:21:39 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 61 for reason Executor for container container_1764236692086_5626_01_000082 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/24 11:21:39 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 30 for reason Executor for container container_1764236692086_5626_01_000040 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/24 11:21:39 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 32 for reason Executor for container container_1764236692086_5